# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Prabhaditya003/FlyRank.ai-internship-work-week1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Classification.** The question "will this content decline?" maps onto the
"Will this one decline / recover?" row of the framing table: a yes/no label,
predicted from an OBSERVED outcome. This isn't ranking many items against each other,
and it isn't unsupervised grouping — the starter dataset already carries an observed
decline flag, so I'm predicting something that already happened and was measured,
not inventing a proxy.

In [ ]:
# Sanity check: task type confirmed by the shape of the label itself.
print("Task type: Classification")
print("Reasoning check -> target is binary, observed, not yet loaded (see Section 4)")

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target: `is_declining_label`** (binary: 1 = declining, 0 = not) — an OBSERVED outcome,
not a rule I'm defining myself.

**Leakage trap:** this label is derived from `trend_direction`, which is computed from
`trend_pct`. That means `trend_direction` and `trend_pct` can never be features — they
ARE the label's formula, not predictors of it. Including either would hand the model
the answer directly. Real features have to come from content/engagement signals that
plausibly precede or correlate with a decline, not signals that define it.

In [ ]:
# Columns that must NEVER be used as features, because they define the label.
LEAKY_COLUMNS = ["trend_direction", "trend_pct"]
TARGET_COLUMN = "is_declining_label"
print(f"Target: {TARGET_COLUMN}")
print(f"Excluded (label-defining) columns: {LEAKY_COLUMNS}")

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: ROC-AUC**, reported alongside **precision/recall against the base rate** of
`is_declining_label`.

Why: an editor uses this output as a prioritized worklist, not a single yes/no cutoff —
ROC-AUC judges whether decliners rank above non-decliners across thresholds, which
matches that use case. Precision/recall vs. base rate keeps the result honest about
whether the model beats just guessing the majority class, especially if declines turn
out to be a minority class (checked below in Section 4).

In [ ]:
# Metric definition (computed for real once data is loaded in Section 4).
PRIMARY_METRIC = "ROC-AUC"
SECONDARY_METRIC = "precision/recall vs. base rate"
print(f"Primary metric: {PRIMARY_METRIC}")
print(f"Secondary metric: {SECONDARY_METRIC}")

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one (pseudonymized) content item, observed over a trailing 90-day window,
for one of 32 clients. `content_id` and `client_id` are pseudonyms — usable only for
grouping and for a grouped train/test split by `client_id` (so one client's items
don't leak across train and test), never as features themselves.

In [ ]:
import pandas as pd

DATA_PATH = "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)
print(df.shape)

# Unit of analysis: one row per content item
unit_of_analysis = df[[
    "content_id", "client_id", "content_type", "word_count",
    "trend_pct", "trend_direction", "is_declining_label"
]].head(10)
unit_of_analysis

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A single if-statement threshold on `trend_pct` would *recover* this exact label — but
that's circular, since it's the label's own definition, not a prediction. The real
decision ("which content should an editor look at before it visibly declines") needs
signals available ahead of or independent from the trend calculation: content
attributes, engagement patterns, keyword/word-count profile, client-level context. Those
signals plausibly interact differently by content_type and by client, and shift over
time — too tangled and too shifting to hand-code as a single rule, which is exactly
where ML earns its place over a fixed rule.

**One-paragraph frame:** For a content editor deciding which items to triage first, we
will build a classification model from historical content and engagement signals,
predicting `is_declining_label` (excluding `trend_pct`/`trend_direction`, which define
the label), measured by ROC-AUC and precision/recall against the base rate. A wrong call
costs either wasted editor time on a false alarm or a real decline going unnoticed. A
plain rule isn't enough because the signals that precede a decline vary by content type
and client and interact in ways too tangled to hand-code. We will claim only
decision-support results — a prioritized worklist, not a guarantee.

In [ ]:
# Base rate check -- also confirms the metric choice in Section 3 makes sense.
target_counts = df["is_declining_label"].value_counts(dropna=False)
target_rate = df["is_declining_label"].mean()
print(target_counts)
print(f"Base rate of decline: {target_rate:.2%}")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.